# Assignment 2B — RAG Pipeline
*Chunking, dense / keyword / hybrid search, reranking, grounded generation and an LLM judge*

**Domain:** Medical and clinical literature (Type 2 Diabetes management)

**Generator:** Assignment 1 model — `mistralai/Mistral-7B-v0.1` + CPT checkpoint + QLoRA adapter

**Variant:** 1 (Default) — Domain Q&A Assistant.

| Part | What we do | Marks |
| --- | --- | --- |
| A | Dense search and chunk size | 5 |
| B | Dense vs BM25 vs hybrid (RRF) | 4 |
| C | Cross-encoder reranking | 4 |
| D | Grounded answers and an LLM judge | 7 |
| **Total** | | **20** |

The final graded run is on the lab server, where the Assignment 1 model files are.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: finds out where the notebook is running (Kubeflow, Colab or local),
# picks a folder to store the files, sets the Hugging Face cache folder and checks the GPU.
# Why it runs first: Hugging Face reads HF_HOME when transformers is imported, so it must be set before.
# Output: environment, device, dtype, GPU name and the folders used.
# ---------------------------------------------------------------------------
import importlib.util
import os
from types import SimpleNamespace


def detect_env():
    """Return "kubeflow", "colab" or "local". Kubeflow is checked first."""
    if ("NB_PREFIX" in os.environ or "KUBERNETES_SERVICE_HOST" in os.environ
            or os.path.isdir("/home/jovyan/data")):
        return "kubeflow"
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "colab"
    except ModuleNotFoundError:      # no "google" package at all, so this is not Colab
        pass
    return "local"


def setup():
    """Find the storage root, point the Hugging Face cache at it, and detect the GPU and dtype."""
    env = detect_env()
    # Storage root per platform (Colab's disk is wiped when the session ends).
    candidates = {"colab": ["/content"], "kubeflow": ["/home/jovyan/assessment", "/home/jovyan/data"],
                  "local": [os.path.expanduser("~/lab-data"), os.getcwd()]}[env]
    root = os.getcwd()
    for path in candidates:          # first folder we can write to wins
        try:
            os.makedirs(path, exist_ok=True)
            probe = os.path.join(path, ".write_test")
            open(probe, "w").close()
            os.remove(probe)
            root = path
            break
        except Exception:
            continue

    # Model cache inside the root. Set before transformers is imported.
    cache_root = os.path.join(root, "hf-cache")
    os.environ["HF_HOME"] = cache_root
    os.environ["HF_HUB_DISABLE_PROGRESS_BAR"] = "1"   # no download bars (they show up empty in the exported notebook)

    # Optional Hugging Face token: use it if it is set (environment variable or Colab secret).
    # This model is public, so the notebook also works without one.
    token = os.environ.get("HF_TOKEN")
    if not token and env == "colab":
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if token:
        os.environ["HF_TOKEN"] = token

    # Device and dtype: GPU if there is one, bf16 when the GPU supports it, else fp16, else fp32 on CPU.
    device, dtype, gpu_name, gpu_mem = "cpu", "float32", None, 0.0
    try:
        import torch
        if torch.cuda.is_available():
            device = "cuda"
            gpu_name = torch.cuda.get_device_name(0)
            gpu_mem = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
            dtype = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
    except Exception:
        pass

    state = SimpleNamespace(ENV=env, ROOT=root, CACHE_ROOT=cache_root, DEVICE=device,
                            DTYPE=dtype, IS_GPU=(device == "cuda"), GPU_NAME=gpu_name, GPU_MEM_GB=gpu_mem)
    print(f"env={env}  device={device}  dtype={dtype}  gpu={gpu_name or 'none'} ({gpu_mem} GB)")
    print(f"storage root : {root}")
    print(f"HF cache     : {cache_root}")
    print(f"HF token     : {'found' if token else 'not set (needed only for gated models such as Llama 3.1)'}")
    return state


L = setup()

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: sets the input and output paths, checks that every input exists and fixes the seed.
# Why: every later step reads these paths, and a missing file should fail here, not halfway through.
# Output: OK or MISSING for each input.
# ---------------------------------------------------------------------------

import json
import random
from pathlib import Path

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ASS1 = Path("../Ass-1")
CORPUS_DIR = ASS1 / "domain_corpus"                      # 14 cleaned .txt files
DATASET_PATH = ASS1 / "instruction_dataset.jsonl"        # 400 rows; the LAST 80 are the eval set
SOURCE_PAIRS_PATH = ASS1 / "instruction_source_pairs.json"  # 50 passages with their pairs
# Do NOT use ASS1 / "instruction_eval.jsonl": it does not match the Assignment 1 data.

# Assignment 1 model files (not in git). Change these if Ass-1 ran from another folder on the lab server.
CPT_DIR = ASS1 / "persistent" / "cpt_ckpt"
ADAPTER_DIR = ASS1 / "persistent" / "adapter"

DATA_DIR, OUT_DIR = Path("data"), Path("outputs")
for folder in (DATA_DIR, OUT_DIR):
    folder.mkdir(exist_ok=True)

# TODO: check each input and print OK / MISSING:
#   - CORPUS_DIR has 14 .txt files
#   - DATASET_PATH has 400 rows
#   - SOURCE_PAIRS_PATH has 50 passages
#   - CPT_DIR and ADAPTER_DIR exist (only on the lab server)

## Prerequisites — Environment Setup

We install the packages that are missing and print their versions, so anyone can see which versions gave these results.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: installs the packages that are missing.
# Why: the notebook should run on a fresh lab server or Colab runtime.
# Output: which packages were already there and which were installed.
# ---------------------------------------------------------------------------

import importlib.util
import subprocess
import sys

# pip name -> import name
PACKAGES = {
    "sentence-transformers": "sentence_transformers",  # embeddings (A, B) and CrossEncoder (C)
    "faiss-cpu": "faiss",                              # IndexFlatIP vector index (A, B)
    "rank-bm25": "rank_bm25",                          # BM25Okapi keyword index (B)
    "transformers": "transformers",                    # tokenizers, generator and judge (A, D)
    "peft": "peft",                                    # QLoRA adapter (D)
    "bitsandbytes": "bitsandbytes",                    # 4-bit loading (D)
    "accelerate": "accelerate",                        # device_map loading (D)
    "pandas": "pandas",                                # tables
}

# TODO: for each package, check importlib.util.find_spec(import_name) and pip install the missing ones
#       with subprocess.run([sys.executable, "-m", "pip", "install", "-q", ...], check=True).

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: prints the Python version, the package versions, torch and the GPU.
# Why: so the output shows which versions and which GPU gave these results.
# Output: a versions table.
# ---------------------------------------------------------------------------

# TODO: print a table with Python, sentence-transformers, faiss, rank_bm25, transformers, peft,
#       bitsandbytes, torch, CUDA and the GPU name (use print_table once it is defined below).

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines print_table(), a small helper that prints aligned tables.
# All reports in this notebook use it. Column widths are calculated from the data,
# so the columns stay aligned even when a value is long.
# ---------------------------------------------------------------------------
def print_table(headers, rows, title=None, align=None, total_row=None):
    """Print an aligned text table.

    headers   : column titles
    rows      : list of tuples, cells already formatted as strings so the caller
                controls thousands separators and dashes for values that don't apply
    title     : optional heading printed above a rule
    align     : per-column "l"/"r"; defaults to label-left, numbers-right
    total_row : optional summary row, printed below a separating rule
    """
    # Default alignment: first column left, the rest right.
    align = align or ["l"] + ["r"] * (len(headers) - 1)
    # Include the total row when working out widths so it fits too.
    body = list(rows) + ([total_row] if total_row else [])
    # Width of each column = longest cell in that column, header included.
    widths = [max(len(headers[c]), max(len(r[c]) for r in body)) for c in range(len(headers))]
    gap = "   "  # space between columns
    # Full table width, used for the ruled lines.
    width = sum(widths) + len(gap) * (len(widths) - 1)

    # Format one row: pad each cell to its column width.
    def line(cells):
        """Format one table row: pad each cell to its column width."""
        return gap.join(c.ljust(w) if a == "l" else c.rjust(w)
                        for c, w, a in zip(cells, widths, align))

    # Print the title, the header, the rows and the optional total row.
    if title:
        print("=" * width)
        print(title)
    print("=" * width)
    print(line(headers))
    print("-" * width)
    for r in rows:
        print(line(r))
    if total_row:
        print("-" * width)
        print(line(total_row))
    print("=" * width)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines md_safe() and working, copied from Assignment 1.
# Why: md_safe prints model answers safely in markdown; working shows progress during slow model loads.
# Output: nothing (definitions only).
# ---------------------------------------------------------------------------
# Helper for markdown tables: puts the model text on one line and escapes the characters that markdown uses for formatting.
def md_safe(text, limit=400):
    """Return model output as one line of text that renders as plain text in markdown."""
    t = " ".join(str(text).split())
    if len(t) > limit:
        t = t[:limit].rstrip() + " ..."
    for ch in "\\`*_[]<>!|#":
        t = t.replace(ch, "\\" + ch)
    return t


# Helper that prints a live status line (time and folder size) for slow steps like loading a 14 GB model,
# so we can see that it is still running.
import threading
import time as _time_mod
from pathlib import Path as _Path


def _folder_gb(folder):
    """Size of everything inside a folder in GB (0 if it does not exist yet)."""
    try:
        return sum(f.stat().st_size for f in _Path(folder).rglob("*") if f.is_file() and not f.is_symlink()) / 1024**3
    except OSError:
        return 0.0


class working:
    """Context manager: shows 'label ... 2m10s' (and the folder size, if given) while the code inside runs."""

    def __init__(self, label, folder=None, expected_gb=None):
        """Remember the message and, optionally, the folder to measure and its expected final size."""
        self.label, self.folder, self.expected_gb = label, folder, expected_gb

    def _line(self):
        """Build the status text for the current moment."""
        secs = int(_time_mod.time() - self.t0)
        text = f"{self.label} ... {secs // 60}m{secs % 60:02d}s"
        if self.folder is not None:
            gb = _folder_gb(self.folder)
            text += f"  ({gb:.1f} of ~{self.expected_gb:.1f} GB)" if self.expected_gb else f"  ({gb:.1f} GB so far)"
        return text

    def _run(self):
        """Background loop: rewrite the status line every 5 seconds until the step ends."""
        while not self._stop.wait(5):
            print("\r" + self._line() + "   ", end="", flush=True)

    def __enter__(self):
        """Start the status line."""
        self.t0, self._stop = _time_mod.time(), threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        """Stop the status line and print the final time."""
        self._stop.set()
        self._thread.join()
        print("\r" + self._line().replace(" ...", " ... done in", 1) + "   ")
        return False

## Before Part A — 10 queries and their answer spans

**Goal:** fix the 10 evaluation queries, their expected answers and their answer spans. Parts A, B and C score every search with these spans.

**What we do:**
1. Take the Assignment 1 eval set: the last 80 rows of `instruction_dataset.jsonl` (80 pairs from 10 passages).
2. Pick one question per eval passage, so the 10 queries cover all 10 passages.
3. Keep each pair's response as the expected answer (used by the judge in D2).
4. Mark the answer span: the shortest exact sentence or phrase in the passage that answers the question.
5. Check that each span appears exactly once in the corpus.

A retrieved chunk is a **hit** if it contains the answer span.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the Assignment 1 eval set and maps each question to its source passage.
# Why: we need the passage to mark the answer span.
# Output: the 10 eval passages with their number of pairs.
# ---------------------------------------------------------------------------

# TODO: load DATASET_PATH, take the last 80 rows (eval_rows).
# TODO: load SOURCE_PAIRS_PATH and map each instruction -> (passage_id, source, passage).
# TODO: assert every eval question is found, and that there are 10 distinct passages.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines the 10 queries, their expected answers and their answer spans.
# Why: the spans are chosen by hand from each passage, so they are written out here.
# Output: data/answer_spans.json and a table of the 10 queries.
# ---------------------------------------------------------------------------

QUERIES = [
    # TODO: one dict per eval passage (10 in total), for example:
    # {"qid": "Q1", "passage_id": 2, "source_file": "hse-ireland-nutrition-dietetic-type2-diabetes-2020.txt",
    #  "query": "...", "expected_answer": "...", "span": "shortest exact sentence from the passage"},
]

def norm(t):
    """Lowercase and collapse whitespace, so line breaks do not break a match."""
    return " ".join(t.lower().split())

def is_hit(chunk_text, span):
    """True if the chunk contains the answer span."""
    return norm(span) in norm(chunk_text)

# TODO: assert len(QUERIES) == 10 and 10 different passage_ids.
# TODO: assert each span occurs exactly once across the corpus (norm(corpus).count(norm(span)) == 1).
# TODO: save QUERIES to DATA_DIR / "answer_spans.json" and show a table: qid, source, query, span, expected answer.

## Part A — Dense search and chunk size (5 marks)

**Goal:** build dense search with FAISS and choose the chunk size for Parts B to D.

**Embedding model.** `all-MiniLM-L6-v2` cuts input at 256 tokens, so 400-token chunks would be truncated. We switch to `BAAI/bge-base-en-v1.5` (512-token limit), so every size fits. Tokens are counted with this model's tokenizer.

**What we do:**
1. A1.1 — one reusable dense search function (FAISS `IndexFlatIP`).
2. A2.1 — chunk the corpus at 100 and 400 tokens (and 500, see the note below), run the 10 queries, fill the table.
3. A2.2 — choose the chunk size.
4. A1.2 — report the setup for the final chunk size.

*Note on the brief:* the A2 text says 100 and 400 tokens but the table header says 500, so we run all three.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the embedding model and prints its limit and vector size.
# Why: the brief asks us to note the model's maximum input length and count tokens with its tokenizer.
# Output: model name, max_seq_length, vector size.
# ---------------------------------------------------------------------------

EMBED_MODEL_ID = "BAAI/bge-base-en-v1.5"
EMBED_BATCH = 64
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "  # TODO: decide to use it or "" and say which

# TODO: embed_model = SentenceTransformer(EMBED_MODEL_ID, device=L.DEVICE); embed_tok = embed_model.tokenizer
# TODO: print EMBED_MODEL_ID, embed_model.max_seq_length, embed_model.get_sentence_embedding_dimension()

### A1.1 Dense search function (1 mark)

One function embeds the chunks, builds a FAISS `IndexFlatIP` index and returns the top k chunks for each query. Embeddings are L2-normalised, so the inner product equals cosine similarity. The index is cached per chunk set, so Parts B and C reuse it.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines dense_search(), the one reusable dense search function.
# Why: Parts A, B and C all call it, so every method is searched the same way.
# Output: nothing (definition only); index build times are stored in BUILD_STATS.
# ---------------------------------------------------------------------------

import time

_INDEX_CACHE, BUILD_STATS = {}, {}

def dense_search(chunks, queries, k, model=None, batch_size=EMBED_BATCH):
    """Embed the chunks, build a FAISS IndexFlatIP, and return the top k chunks for each query.

    chunks  : list of dicts with chunk_id, doc, text, n_tokens
    queries : list of query strings
    returns : one list per query of (rank, chunk_id, score, text), ranks starting at 1
    """
    # TODO 1: key = (id(chunks), len(chunks)); if not cached, time (perf_counter) the encode
    #         (normalize_embeddings=True, batch_size) + faiss.IndexFlatIP(dim) + index.add, store in BUILD_STATS.
    # TODO 2: encode the queries the same way (with QUERY_PREFIX), index.search(q, k).
    # TODO 3: return the ranked lists.
    raise NotImplementedError

### A2.1 Chunk size test (2 marks)

Fixed-size chunks with no overlap, per document, counted with the embedding tokenizer. The text of each chunk is sliced from the original document with the tokenizer's offsets, so the generator sees the original wording.

| Column | How |
| --- | --- |
| Total chunks | number of chunks over the 14 documents |
| Hit rate@3 (%) | share of the 10 queries where one of the top 3 chunks contains the span |
| Tokens sent per query (top 3) | combined `n_tokens` of the top 3, averaged over the 10 queries |

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines chunk_corpus(), fixed-size token chunking with no overlap.
# Why: the brief asks for 100- and 400-token chunks counted with the embedding tokenizer.
# Output: nothing (definition only).
# ---------------------------------------------------------------------------

def chunk_corpus(size, tokenizer, corpus_dir=CORPUS_DIR):
    """Split each document into chunks of `size` tokens (no overlap). Returns a list of chunk dicts."""
    # TODO: for each .txt file: tokenizer(text, add_special_tokens=False, return_offsets_mapping=True),
    #       take ids in windows of `size`, slice the original text from the first to the last offset,
    #       keep {"chunk_id", "doc", "text", "n_tokens"}.
    raise NotImplementedError

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: runs dense search on the 10 queries for each chunk size and fills the A2 table.
# Why: this is the evidence for choosing the chunk size.
# Output: the A2 table (chunk size, total chunks, hit rate@3, tokens per query).
# ---------------------------------------------------------------------------

CHUNK_SIZES = [100, 400, 500]   # 500 covers the typo in the brief's table header
chunk_sets, a2_rows = {}, []

# TODO: for each size: chunks = chunk_corpus(size, embed_tok); assert max n_tokens <= size;
#       results = dense_search(chunks, [q["query"] for q in QUERIES], k=3);
#       hit@3 = share of queries with is_hit(...) in the top 3; tokens = mean of the top-3 n_tokens sums.
# TODO: print_table(["Chunk size (tokens)", "Total chunks", "Hit rate@3 (%)", "Tokens sent per query (top 3)"], a2_rows)

### A2.2 Chosen chunk size (1 mark)

*Inference cell below: two or three sentences built from the A2 numbers. Weigh hit rate, tokens per query, and the generator's context: the Assignment 1 adapter was trained with `SFT_MAX_LEN = 512`, and the model's context window is 4,096 tokens.*

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: sets the chosen chunk size and measures the Part D prompt length with the Mistral tokenizer.
# Why: the generator counts tokens differently from the embedding model.
# Output: the chosen size and the prompt length in Mistral tokens.
# ---------------------------------------------------------------------------

CHUNK_SIZE = None   # TODO: set from the A2 table
# TODO: chunks = chunk_sets[CHUNK_SIZE]
# TODO: load the Mistral tokenizer from ADAPTER_DIR and count the tokens of one full Part D prompt (3 chunks).

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: Inference - A2.2: the chosen chunk size and why (numbers come from the A2 table).
# Why: the reason must match the numbers, so the text is built from them.
# Output: two or three sentences.
# ---------------------------------------------------------------------------

from IPython.display import Markdown, display
# TODO: display(Markdown(f"... {CHUNK_SIZE} ... hit rate ... tokens ... 512 / 4,096 ..."))

### A1.2 Dense search setup (1 mark)

Reported for the final chunk set chosen in A2.2.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: prints the dense search setup for the final chunk set.
# Why: the brief asks for these numbers for the final chunk set.
# Output: model name, vector size, similarity, batch size, number of chunks, index build time.
# ---------------------------------------------------------------------------

# TODO: print_table(["Setting", "Value"], [
#     ("Embedding model", EMBED_MODEL_ID), ("Vector size", ...), ("Similarity", "cosine (inner product on L2-normalised vectors)"),
#     ("Batch size", str(EMBED_BATCH)), ("Chunks", ...), ("Index build time (s)", ...)])

## Part B — Keyword and hybrid search (4 marks)

**Goal:** compare dense search, BM25 and a hybrid of the two on the same 10 queries, top 5 each.

**What we do:**
1. B1 — dense search from Part A.
2. B2 — BM25 index, top 5.
3. B3 — hybrid with Reciprocal Rank Fusion over the top 20 of each method:
   `RRF(d) = 1 / (60 + rank in dense) + 1 / (60 + rank in BM25)`. Ranks start at 1; a chunk missing from a list gets nothing from it.
4. B4 — which method works best, with one example query.

| Metric | How |
| --- | --- |
| Query latency (ms) | time to return the top 5 for one query, averaged over the 10 queries, after one warm-up run |
| Hit rate@1 (%) | share of queries whose first result contains the span |
| Hit rate@3 (%) | share of queries with the span in the top 3 |

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines the shared evaluate() helper used for every method in Parts B and C.
# Why: one helper means the rows of the table are measured the same way.
# Output: nothing (definition only).
# ---------------------------------------------------------------------------

def evaluate(search_fn, k=5):
    """Run search_fn(query, k) on the 10 queries after one warm-up.

    Returns avg latency (ms), hit rate@1 (%), hit rate@3 (%) and the ranked lists per query.
    """
    # TODO: warm-up call (not timed); time each query with time.perf_counter(); score with is_hit.
    raise NotImplementedError

### B1 Dense (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: runs dense search (top 5) through evaluate().
# Why: fills the Dense row of the B table.
# Output: latency, hit rate@1, hit rate@3 for dense.
# ---------------------------------------------------------------------------

# TODO: dense_fn = lambda q, k: dense_search(chunks, [q], k)[0]
# TODO: b_results["Dense"] = evaluate(dense_fn)

### B2 BM25 (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: builds a BM25 index over the chunks and runs it (top 5).
# Why: keyword search catches exact terms such as drug names and HbA1c values.
# Output: latency, hit rate@1, hit rate@3 for BM25.
# ---------------------------------------------------------------------------

import re
from rank_bm25 import BM25Okapi

def bm25_tokens(text):
    """Lowercase word tokens; keeps terms like hba1c and sglt2 whole."""
    return re.findall(r"[a-z0-9]+", text.lower())

# TODO: bm25 = BM25Okapi([bm25_tokens(c["text"]) for c in chunks])
# TODO: bm25_fn(q, k): scores = bm25.get_scores(bm25_tokens(q)); top = np.argsort(-scores)[:k]
# TODO: b_results["BM25"] = evaluate(bm25_fn)

### B3 Hybrid with RRF (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines rrf_fuse(), tests it on a toy case, and runs the hybrid search (top 5).
# Why: RRF combines the two ranked lists without needing comparable scores.
# Output: the toy test result, then latency, hit rate@1, hit rate@3 for hybrid.
# ---------------------------------------------------------------------------

def rrf_fuse(dense_ids, bm25_ids, k=60, top=5):
    """Reciprocal Rank Fusion of two ranked id lists (ranks start at 1). Returns the top `top` ids."""
    # TODO: score[d] += 1 / (k + rank) for each list; sort by score, return the first `top`.
    raise NotImplementedError

# TODO: toy test: a chunk ranked 2nd and 3rd must beat a chunk ranked 1st in only one list.
# TODO: hybrid_fn(q, k): top 20 from dense and from BM25, then rrf_fuse(..., top=k)
# TODO: b_results["Hybrid (RRF)"] = evaluate(hybrid_fn)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: prints the B comparison table and the rank of the answer chunk per query and method.
# Why: the table is graded; the per-query ranks are the evidence for B4.
# Output: the B table and a per-query rank table.
# ---------------------------------------------------------------------------

# TODO: print_table(["Method", "Average query latency (ms)", "Hit rate@1 (%)", "Hit rate@3 (%)"], rows)
# TODO: per query: rank of the first hit under Dense, BM25 and Hybrid (or "-" if not in the top 5)

### B4 Best method (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: Inference - B4: which method works best and why, with one example query.
# Why: the text is built from the B table, so it stays correct for the real numbers.
# Output: two or three sentences naming the method and one query as evidence.
# ---------------------------------------------------------------------------

BEST_METHOD = None   # TODO: "Dense", "BM25" or "Hybrid (RRF)", from the table
# TODO: display(Markdown(f"... {BEST_METHOD} ... example query ... ranks ..."))

## Part C — Reranking (4 marks)

**Goal:** re-score the top 10 of the best Part B method with a cross-encoder and keep the top 3.

**What we do:**
1. C1 — retrieve the top 10 with the best method; re-score with `cross-encoder/ms-marco-MiniLM-L-6-v2`; keep the top 3.
2. C2 — show the top 3 before and after reranking for each query, and the rank change rate.
3. C3 — reranking time per query and hit rate@1 before and after.
4. C4 — did reranking help?

**Rank change rate** = share of the 10 queries whose first result changes after reranking.

### C1 Rerank (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the cross-encoder and reranks the top 10 of the best method for each query.
# Why: a cross-encoder reads the query and the chunk together, so it can order them better.
# Output: the device used and final_chunks (top 3 per query).
# ---------------------------------------------------------------------------

RERANK_MODEL_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"
# TODO: ce = CrossEncoder(RERANK_MODEL_ID, max_length=512, device=L.DEVICE)
# TODO: for each query: top10 = best_fn(query, 10); scores = ce.predict([(query, c_text) for ...])
#       keep the 3 highest -> final_chunks[qid]; keep before_top3[qid] = top10[:3]
# TODO: time only ce.predict (warm-up first; torch.cuda.synchronize() before stopping the timer on GPU)
final_chunks, before_top3, rerank_ms = {}, {}, []

### C2 Compare rankings (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: shows the top 3 before and after reranking for each query.
# Why: the brief asks to see the change for every query.
# Output: a table per query (chunk id, first 80 characters, hit marker) and the rank change rate.
# ---------------------------------------------------------------------------

# TODO: for each query, print before and after top 3 with md_safe(text[:80]) and a hit marker.
# TODO: rank_change_rate = share of queries where before_top3[qid][0] != final_chunks[qid][0]

### C3 Measure (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: fills the reranking table.
# Why: weighs accuracy against the time reranking adds.
# Output: reranking time per query, rank change rate, hit rate@1 before and after.
# ---------------------------------------------------------------------------

# TODO: print_table(["Measure", "Value"], [
#     ("Reranking time per query (ms)", ...), ("Rank change rate (%)", ...),
#     ("Hit rate@1 before reranking (from Part B)", ...),   # must equal the Part B value for BEST_METHOD
#     ("Hit rate@1 after reranking", ...)])

### C4 Did reranking help? (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: Inference - C4: whether reranking helped, from the hit rates and the time added.
# Why: the text is built from the C3 numbers.
# Output: two or three sentences.
# ---------------------------------------------------------------------------

# TODO: display(Markdown(f"Hit rate@1 went from ... to ... for +... ms per query, so ..."))

## Part D — Generate and check answers (7 marks)

### D1 Generate grounded answers (3 marks)

**Goal:** answer the 10 queries with the Assignment 1 model, using only the 3 reranked chunks, with citations and a "Not found" reply.

**What we do:**
1. D1.1 — write and show the grounded prompt.
2. D1.2 — answer the 10 queries with the 3 reranked chunks, labelled [1] to [3].
3. D1.3 — one query whose answer is not in the corpus.

If the Assignment 1 model ignores the chunks or never says "Not found", we switch to an instruct model, name it, and show one example of the problem.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the Assignment 1 generator: CPT model in 4-bit plus the QLoRA adapter.
# Why: the brief asks us to reuse the Assignment 1 model as the answer generator.
# Output: 4-bit layers, memory footprint and the generator name.
# ---------------------------------------------------------------------------

GEN_BASE_ID = "mistralai/Mistral-7B-v0.1"
GENERATOR_NAME = f"{GEN_BASE_ID} + CPT + QLoRA adapter (Assignment 1)"
# TODO: bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
#                                       bnb_4bit_compute_dtype=..., bnb_4bit_use_double_quant=True)
# TODO: with working("Loading the generator"):
#           gen_base = AutoModelForCausalLM.from_pretrained(CPT_DIR, quantization_config=bnb_config, device_map={"": L.DEVICE})
#       gen_model = PeftModel.from_pretrained(gen_base, ADAPTER_DIR).eval()
# TODO: gen_tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)   # carries the Instruction/Response chat template

#### D1.1 Generation prompt (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines the grounded prompt and prints it.
# Why: the brief asks to show the prompt; the same template is used for every query.
# Output: the prompt template.
# ---------------------------------------------------------------------------

GEN_PROMPT = """Answer the question using ONLY the numbered chunks below.
Cite the chunk number you used after each fact, for example [2].
If the chunks do not contain the answer, reply exactly: Not found in the documents

Chunks:
[1] {chunk_1}
[2] {chunk_2}
[3] {chunk_3}

Question: {question}"""
print(GEN_PROMPT)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines generate_grounded(), adapted from generate_instruct() in Assignment 1.
# Why: the prompt goes into the user turn so the model sees Instruction: ... Response: as in training.
# Output: nothing (definition only).
# ---------------------------------------------------------------------------

def generate_grounded(question, chunk_texts, max_new_tokens=150):
    """Fill GEN_PROMPT with the 3 chunks, generate greedily, and return the answer text."""
    # TODO: prompt = gen_tok.apply_chat_template([{"role": "user", "content": GEN_PROMPT.format(...)}],
    #                                            tokenize=False, add_generation_prompt=True)
    # TODO: generate with do_sample=False; decode only the new tokens; cut at "Instruction:".
    raise NotImplementedError

#### D1.2 All 10 answers (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers the 10 queries with their 3 reranked chunks.
# Why: these answers are what the judge checks in D2.
# Output: a table: query, answer, chunks cited.
# ---------------------------------------------------------------------------

answers = {}
# TODO: for each query: answers[qid] = generate_grounded(query, [c_text for c in final_chunks[qid]])
# TODO: show the table with md_safe(); list the [n] citations found in each answer.

#### D1.3 A query whose answer is not in the corpus (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: runs one out-of-corpus question through the same retrieval, reranking and prompt.
# Why: shows that the model replies Not found in the documents instead of guessing.
# Output: the question, its 3 chunks and the reply.
# ---------------------------------------------------------------------------

OOC_QUERY = None   # TODO: a question the 14 documents cannot answer
# TODO: retrieve with BEST_METHOD, rerank, generate; show the 3 chunks and the reply.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: Inference - D1: did the generator stay grounded? (built from the answers above).
# Why: the brief asks to name the generator, and to show one example if we switched models.
# Output: the generator used, citation counts, the Not-found result, and the failure example if any.
# ---------------------------------------------------------------------------

# TODO: count answers with a citation or an exact "Not found in the documents".
# TODO: if the Assignment 1 model failed: show one bad answer next to its chunks, switch to
#       Qwen/Qwen2.5-7B-Instruct or meta-llama/Llama-3.1-8B-Instruct, set GENERATOR_NAME, and rerun D1.2-D1.3.

### D2 Check the answers with an LLM judge (4 marks)

**Goal:** grade each answer for correctness against the expected answer and for support by its 3 chunks.

**Judge model:** different from the generator. Mistral generator → `Qwen/Qwen2.5-7B-Instruct`; if the generator was switched to Qwen → `meta-llama/Llama-3.1-8B-Instruct` (gated, needs `HF_TOKEN`). Greedy decoding (temperature 0).

**What we do:**
1. D2.1 — run the judge on the 10 answers; per-query table and summary counts.
2. D2.2 — spot-check 3 verdicts by hand.
3. D2.3 — find the cause of one wrong or unsupported answer.
4. D2.4 — suggest one fix.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: frees the generator and loads the judge model.
# Why: two 7B models do not need to share the GPU at the same time.
# Output: the judge name and memory footprint.
# ---------------------------------------------------------------------------

JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"   # TODO: switch to meta-llama/Llama-3.1-8B-Instruct if the generator is Qwen
assert JUDGE_MODEL_ID not in GENERATOR_NAME, "the judge must be a different model from the generator"
# TODO: del gen_model, gen_base; gc.collect(); torch.cuda.empty_cache()
# TODO: load the judge in 4-bit with its own tokenizer and chat template.

#### D2.1 Judge prompt and results (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines the judge prompt and prints it.
# Why: the brief asks to show the prompt; it follows every hint in the brief.
# Output: the judge prompt.
# ---------------------------------------------------------------------------

JUDGE_PROMPT = """You are grading one answer about Type 2 Diabetes.

Question: {question}
Expected answer (reference): {expected}
Retrieved chunks:
[1] {chunk_1}
[2] {chunk_2}
[3] {chunk_3}
Model answer: {answer}

Check 1, CORRECT: compare the model answer with the expected answer.
  CORRECT = it states the same key fact as the expected answer.
  PARTLY CORRECT = it states only part of the key fact, or the right fact with a detail missing or wrong.
  WRONG = it states a different fact, contradicts the expected answer, or says "Not found in the documents".
Check 2, SUPPORTED: can every statement in the model answer be found in the 3 chunks above?
  YES = every statement is in the chunks. NO = at least one statement is not.
  A "Not found in the documents" reply makes no claims, so it is SUPPORTED: YES.
REASON: one sentence. If SUPPORTED is NO, quote the unsupported statement.

Reply in exactly this format, with nothing else:
CORRECT: <CORRECT | PARTLY CORRECT | WRONG>
SUPPORTED: <YES | NO>
REASON: <one sentence>"""
print(JUDGE_PROMPT)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: runs the judge on the 10 answers and parses the verdicts.
# Why: a fixed output format lets us read the verdicts with code.
# Output: the per-query table (query, correct, supported, reason) and the summary counts.
# ---------------------------------------------------------------------------

def parse_verdict(text):
    """Read CORRECT, SUPPORTED and REASON lines; returns None for a field that did not parse."""
    # TODO: re.search(r"^CORRECT:\s*(CORRECT|PARTLY CORRECT|WRONG)", text, re.M) and the same for the others.
    raise NotImplementedError

verdicts = {}
# TODO: for each qid: fill JUDGE_PROMPT, generate with do_sample=False, verdicts[qid] = parse_verdict(output)
# TODO: print the raw output for any verdict that failed to parse.
# TODO: per-query table: query, correct, supported, reason
# TODO: print_table(["Check", "Count out of 10"], [("Correct", ...), ("Partly correct", ...),
#                                                   ("Wrong", ...), ("Supported by the chunks", ...)])

#### D2.2 Spot-check the judge (1 mark)

*Pick 3 answers (ideally with different verdicts), read each answer and its 3 chunks, and write your own CORRECT and SUPPORTED. Say how many of the 3 judge verdicts you agree with, and explain one disagreement if there is one.*

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: shows 3 answers with their chunks and the judge's verdicts for the manual check.
# Why: the spot-check must be done by reading the chunks ourselves.
# Output: 3 answers, their chunks, the judge verdict and our own verdict.
# ---------------------------------------------------------------------------

SPOT_CHECK = {
    # TODO: qid: {"my_correct": "...", "my_supported": "...", "note": "..."},
}
# TODO: show each answer, its 3 chunks, the judge verdict and SPOT_CHECK[qid]; count agreements out of 3.

#### D2.3 Find the cause (1 mark)

*Pick one answer marked WRONG or SUPPORTED: NO (if there is none, pick the weakest PARTLY CORRECT answer and say so). Show its 3 chunks and decide: **retrieval** (the needed fact is not in the chunks) or **generation** (the fact is in a chunk but the model did not use it, or added something).*

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: shows the chosen failed answer with its 3 chunks and checks whether the answer span was retrieved.
# Why: is_hit on the span tells us whether retrieval or generation caused the problem.
# Output: the answer, its 3 chunks, span found yes/no, and the cause.
# ---------------------------------------------------------------------------

FAILED_QID = None   # TODO
# TODO: show the answer and its 3 chunks; span_found = any(is_hit(c_text, span) for the 3 chunks)
# TODO: cause = "generation" if span_found else "retrieval"

#### D2.4 Suggest a fix (1 mark)

*One change that matches the cause, and why it would help: a different chunk size or overlap (span split), hybrid or BM25 search (exact term missed), more candidates or more chunks (fact ranked 4th or 5th), or stricter prompt wording or an instruct model (content ignored or added).*

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: Inference - D2: judge summary, spot-check agreement, the cause and the suggested fix.
# Why: the text is built from the D2 results, so it matches the real numbers.
# Output: a short summary of D2.
# ---------------------------------------------------------------------------

# TODO: display(Markdown(f"The judge ({JUDGE_MODEL_ID}) marked ... correct ... We agree with .../3 ... "
#                        f"The failure in {FAILED_QID} was caused by {cause} ... Fix: ..."))

## Submission

1. Restart the kernel and run all cells on the lab server.
2. Check that every output is visible.
3. Export: `jupyter nbconvert --to html Ass-2/Assignment2B.ipynb`.

The full checklist is in `Ass-2/Assignment-2B-roadmap.md` (Step 8).